# DeepCache — Stable Diffusion 1.5 no Google Colab (GPU T4)

Projeto individual. Reproduz o baseline do artigo DeepCache (Ma et al., CVPR 2024) e testa uma intervenção no critério de atualização do cache da U-Net.

Configuração comum às três versões: `runwayml/stable-diffusion-v1-5`, float16, 50 passos PLMS (PNDM com `skip_prk_steps=True`), guidance 7.5, 512×512, seed 42.

1. Stable Diffusion v1.5 sem cache.
2. DeepCache oficial (`DeepCacheSDHelper`, `cache_interval=3`, `cache_branch_id=0`).
3. DeepCache adaptativo: a U-Net completa é recalculada somente quando a distância relativa L2 entre o latent atual e o último latent recalculado passa de `tau`.

Fluxo: instalação e reinício, verificação do ambiente, prompts, scheduler, intervenção, calibração de `tau` com 3 prompts exclusivos e avaliação final nos 5 prompts oficiais.

## Bloco 1 — Instalação e ambiente

In [ ]:
# Instale dependências sem reinstalar o PyTorch/CUDA já configurado pelo Colab.
%pip install -q 'diffusers>=0.43,<0.44' 'transformers>=5.24,<6' 'huggingface-hub>=1.32,<2' 'tokenizers>=0.23.1,<0.24' 'sentence-transformers>=6.3,<7' 'accelerate>=1.12' 'safetensors>=0.7' jedi lpips torchmetrics DeepCache

print('Instalação concluída. Reinicie o runtime (Ambiente de execução > Reiniciar sessão) e execute o notebook a partir do Bloco 1 de verificação.')

In [ ]:
# Execute após reiniciar o runtime do Colab.
import importlib.metadata
import platform
import torch

print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
print('Diffusers:', importlib.metadata.version('diffusers'))
print('Transformers:', importlib.metadata.version('transformers'))
print('Hugging Face Hub:', importlib.metadata.version('huggingface-hub'))
print('Tokenizers:', importlib.metadata.version('tokenizers'))
print('CUDA disponível:', torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError('Ative uma GPU NVIDIA no Google Colab: Ambiente de execução > Alterar tipo de ambiente de execução > GPU T4')

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print('GPU:', gpu_name)
print(f'VRAM total: {vram_gb:.2f} GB')
if vram_gb < 14:
    print('AVISO: a execução pode dar OOM em GPUs com menos de 15 GB.')

## Bloco 2 — Prompts e configuração

Os 5 prompts oficiais são copiados de `data/prompts_geracao.txt` e embutidos aqui, para o notebook rodar sem clonar o repositório. Os 3 prompts de calibração são exclusivos e nunca entram na avaliação final.

In [ ]:
from pathlib import Path

OFFICIAL_PROMPTS = [
    'A futuristic cyberpunk city at night with neon reflections on wet streets, photorealistic, 8k',
    'A golden retriever puppy wearing a tiny red astronaut helmet on the moon, studio lighting',
    'An oil painting of a cozy wooden cabin in a snowy pine forest at sunset, warm light in windows',
    'A detailed macro photograph of a mechanical watch mechanism with golden gears and blue sapphires',
    'A serene Japanese zen garden with cherry blossom trees, koi pond, and wooden bridge in spring',
]

CALIBRATION_PROMPTS = [
    'A lighthouse on a rocky coast during a stormy night, long exposure photograph',
    'A bowl of ramen with soft-boiled egg and green onions on a wooden table, food photography',
    'A knight in ornate silver armor standing in a misty forest, digital painting',
]

assert len(OFFICIAL_PROMPTS) == 5, f'Esperado 5 prompts oficiais, encontrado {len(OFFICIAL_PROMPTS)}'
assert len(CALIBRATION_PROMPTS) == 3, f'Esperado 3 prompts de calibração, encontrado {len(CALIBRATION_PROMPTS)}'
assert not set(OFFICIAL_PROMPTS) & set(CALIBRATION_PROMPTS), 'Prompts de calibração e de avaliação se sobrepõem'

PROJECT_DIR = Path('/content/deepcache-experiment')
RESULTS_DIR = PROJECT_DIR / 'results'
prompts_file = PROJECT_DIR / 'data' / 'prompts_geracao.txt'
if prompts_file.exists():
    file_prompts = [line.strip() for line in prompts_file.read_text(encoding='utf-8').splitlines() if line.strip()]
    assert file_prompts == OFFICIAL_PROMPTS, 'data/prompts_geracao.txt difere da lista embutida no notebook'
    print('Prompts conferidos com data/prompts_geracao.txt')
else:
    print('data/prompts_geracao.txt não encontrado. Usando a lista embutida (5 prompts).')
print(f'Prompts oficiais: {len(OFFICIAL_PROMPTS)} | Prompts de calibração: {len(CALIBRATION_PROMPTS)}')

## Bloco 3 — Pipeline e scheduler PLMS

O artigo usa 50 passos PLMS no Stable Diffusion. Aqui o PLMS é o `PNDMScheduler` com `skip_prk_steps=True`. Se esse scheduler gerar timesteps duplicados, o notebook troca para `LMSDiscreteScheduler`, que não repete timesteps. A célula valida a unicidade antes de qualquer medição.

In [ ]:
import gc
import time
import numpy as np
import torch
from diffusers import StableDiffusionPipeline, PNDMScheduler, LMSDiscreteScheduler

MODEL_ID = 'runwayml/stable-diffusion-v1-5'
SEED = 42
STEPS = 50
GUIDANCE = 7.5
WIDTH = 512
HEIGHT = 512
CACHE_INTERVAL = 3
CACHE_BRANCH_ID = 0
ADAPTIVE_CACHE_INTERVAL = STEPS + 1  # nunca dispara o intervalo fixo; a decisão é adaptativa
TAU_CANDIDATES = [0.015, 0.020, 0.030]
MAX_CLIP_DROP = 0.01  # queda máxima aceitável de CLIP médio vs. sem cache na calibração
TAU_FIXED = None  # defina um float (ex.: 0.020) para pular a calibração

assert torch.cuda.is_available(), 'Ative a GPU T4: Ambiente de execução > Alterar tipo de ambiente de execução > GPU T4'

pipe = StableDiffusionPipeline.from_pretrained(MODEL_ID, dtype=torch.float16).to('cuda')
pipe.enable_attention_slicing('max')

pipe.scheduler = PNDMScheduler.from_config(pipe.scheduler.config, skip_prk_steps=True)
pipe.scheduler.set_timesteps(STEPS)
if len(pipe.scheduler.timesteps.tolist()) != len(set(pipe.scheduler.timesteps.tolist())):
    print('PNDM gerou timesteps duplicados. Usando LMSDiscreteScheduler como fallback.')
    pipe.scheduler = LMSDiscreteScheduler.from_config(pipe.scheduler.config)
    pipe.scheduler.set_timesteps(STEPS)

ts = pipe.scheduler.timesteps.tolist()
assert len(ts) == len(set(ts)), f'Timesteps duplicados detectados: {ts}'
print(f'Scheduler validado: {len(ts)} timesteps únicos.')
SCHEDULER_NAME = type(pipe.scheduler).__name__
print('Scheduler em uso:', SCHEDULER_NAME)

## Bloco 4 — DeepCache oficial com ponto de decisão adaptativo

`DeepCacheSDHelper.is_skip_step(block_i, layer_i, blocktype)` decide, para cada bloco e cada passo, se o bloco é recalculado ou se reaproveita `cached_output`. A subclasse abaixo mantém toda a lógica do pacote oficial e troca apenas essa decisão quando `tau` é definido. Um callback de fim de passo calcula a distância relativa L2 e define `refresh`, que vale para a próxima chamada da U-Net.

In [ ]:
from DeepCache import DeepCacheSDHelper


class AdaptiveDeepCacheSDHelper(DeepCacheSDHelper):
    '''DeepCacheSDHelper com a decisão de atualização do cache substituída.

    tau=None: comportamento oficial, com intervalo fixo definido por cache_interval.
    tau=float: a U-Net é recalculada inteira somente quando refresh=True.
    Nos demais passos, os blocos reaproveitam cached_output pela lógica oficial.
    '''

    def __init__(self, pipe=None, tau=None):
        super().__init__(pipe=pipe)
        self.tau = tau

    def reset_states(self):
        super().reset_states()
        self.refresh = True
        self.ref_latent = None
        self.distances = []
        self.called_steps = set()
        self.full_steps = set()

    def is_skip_step(self, block_i, layer_i, blocktype='down'):
        if self.tau is None:
            skip = super().is_skip_step(block_i, layer_i, blocktype)
        elif self.refresh:
            skip = False
        else:
            skip = super().is_skip_step(block_i, layer_i, blocktype)

        # O bloco (0, 0) 'down' é consultado uma vez por passo. Contamos por cur_timestep,
        # que é único por chamada da U-Net, para não contar duas vezes o bloco e sua atenção.
        if block_i == 0 and layer_i == 0 and blocktype == 'down':
            self.called_steps.add(self.cur_timestep)
            if not skip:
                self.full_steps.add(self.cur_timestep)
        return skip


def make_adaptive_callback(helper):
    def _on_step_end(pipe, step, timestep, callback_kwargs):
        z = callback_kwargs['latents'].detach().float()
        if helper.ref_latent is None:
            distance = float('inf')
        else:
            distance = (
                torch.linalg.vector_norm(z - helper.ref_latent)
                / (torch.linalg.vector_norm(helper.ref_latent) + 1e-8)
            ).item()
        helper.distances.append(distance)
        helper.refresh = distance > helper.tau
        if helper.refresh:
            helper.ref_latent = z.clone()
        return callback_kwargs
    return _on_step_end


def make_adaptive_helper(tau):
    helper = AdaptiveDeepCacheSDHelper(pipe=pipe, tau=tau)
    helper.set_params(cache_interval=ADAPTIVE_CACHE_INTERVAL, cache_branch_id=CACHE_BRANCH_ID)
    return helper


def reset_seed():
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)


def generate(prompt, helper=None):
    '''Gera uma imagem. Retorna (imagem, latência em s, pico de VRAM em MB, passos completos da U-Net).'''
    reset_seed()
    generator = torch.Generator(device='cuda').manual_seed(SEED)
    kwargs = dict(
        prompt=prompt,
        num_inference_steps=STEPS,
        guidance_scale=GUIDANCE,
        width=WIDTH,
        height=HEIGHT,
        generator=generator,
        output_type='pil',
    )
    if helper is not None:
        helper.enable()
    try:
        if helper is not None and helper.tau is not None:
            kwargs['callback_on_step_end'] = make_adaptive_callback(helper)
            kwargs['callback_on_step_end_tensor_inputs'] = ['latents']
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()
        start = time.perf_counter()
        image = pipe(**kwargs).images[0]
        torch.cuda.synchronize()
        elapsed = time.perf_counter() - start
        peak_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)
        if helper is None:
            full_steps = STEPS
        else:
            assert len(helper.called_steps) == STEPS, f'Interceptação incompleta: {len(helper.called_steps)} de {STEPS} passos'
            full_steps = len(helper.full_steps)
    finally:
        if helper is not None:
            helper.disable()
    return image, elapsed, peak_mb, full_steps

## Bloco 5 — Calibração de tau com prompts exclusivos

Cada `tau` candidato roda nos 3 prompts de calibração. O critério escolhe o `tau` que economiza mais passos completos da U-Net sem queda de CLIP Score acima de `MAX_CLIP_DROP` em relação ao Stable Diffusion sem cache. Se nenhum `tau` atender ao critério, o notebook usa o de menor queda de CLIP. Para fixar um valor, preencha `TAU_FIXED` no Bloco 3.

In [ ]:
from transformers import CLIPModel, CLIPProcessor

clip_model = CLIPModel.from_pretrained('openai/clip-vit-base-patch32').eval().to('cuda')
clip_processor = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')


@torch.no_grad()
def clip_score(prompt, image):
    inputs = clip_processor(text=[prompt], images=image, return_tensors='pt', padding=True).to('cuda')
    out = clip_model(**inputs)
    img_emb = out.image_embeds / out.image_embeds.norm(dim=-1, keepdim=True)
    txt_emb = out.text_embeds / out.text_embeds.norm(dim=-1, keepdim=True)
    return (img_emb @ txt_emb.T).item()


# Aquecimento: a primeira geração carrega kernels CUDA e não entra nas medições.
_ = generate(CALIBRATION_PROMPTS[0])

calib_base_clips, calib_base_times = [], []
for prompt in CALIBRATION_PROMPTS:
    image, elapsed, _, _ = generate(prompt)
    calib_base_clips.append(clip_score(prompt, image))
    calib_base_times.append(elapsed)
calib_base_clip = float(np.mean(calib_base_clips))

calibration_rows = []
for tau in TAU_CANDIDATES:
    helper = make_adaptive_helper(tau)
    clips, times, fulls = [], [], []
    for prompt in CALIBRATION_PROMPTS:
        image, elapsed, _, full_steps = generate(prompt, helper)
        clips.append(clip_score(prompt, image))
        times.append(elapsed)
        fulls.append(full_steps)
    calibration_rows.append({
        'tau': tau,
        'passos_completos': float(np.mean(fulls)),
        'passos_economizados': float(STEPS - np.mean(fulls)),
        'latencia_s': float(np.mean(times)),
        'clip': float(np.mean(clips)),
        'queda_clip': calib_base_clip - float(np.mean(clips)),
    })

feasible = [row for row in calibration_rows if row['queda_clip'] <= MAX_CLIP_DROP]
if TAU_FIXED is not None:
    TAU = TAU_FIXED
    print(f'tau fixado manualmente: {TAU}')
elif feasible:
    best = max(feasible, key=lambda row: (row['passos_economizados'], -row['queda_clip']))
    TAU = best['tau']
else:
    best = min(calibration_rows, key=lambda row: row['queda_clip'])
    TAU = best['tau']
    print('Nenhum tau atingiu a queda máxima de CLIP. Usando o de menor queda.')

print('Calibração (prompts exclusivos):')
print('| tau | Passos completos (de 50) | Passos economizados | Latência média (s) | CLIP médio | Queda de CLIP vs. sem cache |')
print('|---:|---:|---:|---:|---:|---:|')
for row in calibration_rows:
    print('| {:.3f} | {:.1f} | {:.1f} | {:.3f} | {:.4f} | {:.4f} |'.format(
        row['tau'], row['passos_completos'], row['passos_economizados'],
        row['latencia_s'], row['clip'], row['queda_clip']))
print('Sem cache na calibração: CLIP médio {:.4f}, latência média {:.3f} s'.format(
    calib_base_clip, float(np.mean(calib_base_times))))
print(f'tau escolhido para a avaliação final: {TAU}')

## Bloco 6 — Avaliação final nos 5 prompts oficiais

Cada prompt é gerado nas três versões com a mesma seed. As imagens são salvas em `results/`.

In [ ]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
official_helper = AdaptiveDeepCacheSDHelper(pipe=pipe, tau=None)
official_helper.set_params(cache_interval=CACHE_INTERVAL, cache_branch_id=CACHE_BRANCH_ID)
adaptive_helper = make_adaptive_helper(TAU)

records = []
for idx, prompt in enumerate(OFFICIAL_PROMPTS, start=1):
    b_img, b_t, b_vram, b_full = generate(prompt)
    o_img, o_t, o_vram, o_full = generate(prompt, official_helper)
    a_img, a_t, a_vram, a_full = generate(prompt, adaptive_helper)
    b_img.save(RESULTS_DIR / f'prompt_{idx:02d}_sem_cache.png')
    o_img.save(RESULTS_DIR / f'prompt_{idx:02d}_deepcache_oficial.png')
    a_img.save(RESULTS_DIR / f'prompt_{idx:02d}_deepcache_adaptativo.png')
    records.append({
        'prompt': prompt,
        'images': (b_img, o_img, a_img),
        'time': (b_t, o_t, a_t),
        'vram': (b_vram, o_vram, a_vram),
        'full': (b_full, o_full, a_full),
        'clip': (clip_score(prompt, b_img), clip_score(prompt, o_img), clip_score(prompt, a_img)),
    })
    print(f'{idx:02d}/5 | sem cache={b_t:.2f}s | oficial={o_t:.2f}s ({o_full} passos completos) | adaptativo={a_t:.2f}s ({a_full} passos completos)')

## Bloco 7 — LPIPS, tabela final e exportação

O pipeline é liberado antes de carregar o LPIPS. O LPIPS compara cada imagem com a versão sem cache, redimensionada para 256×256.

In [ ]:
import csv
import json
import lpips
from PIL import Image, ImageDraw

del pipe, official_helper, adaptive_helper
gc.collect()
torch.cuda.empty_cache()

lpips_metric = lpips.LPIPS(net='alex').eval().cpu()


def pil_to_lpips_tensor(image):
    array = np.asarray(image.resize((256, 256))).astype(np.float32) / 127.5 - 1.0
    return torch.from_numpy(array).permute(2, 0, 1).unsqueeze(0)


@torch.no_grad()
def lpips_vs_baseline(baseline, candidate):
    return lpips_metric(pil_to_lpips_tensor(baseline), pil_to_lpips_tensor(candidate)).item()


for rec in records:
    b_img, o_img, a_img = rec['images']
    rec['lpips'] = (0.0, lpips_vs_baseline(b_img, o_img), lpips_vs_baseline(b_img, a_img))

names = ['Stable Diffusion sem cache', 'DeepCache oficial', 'DeepCache adaptativo (tau={:.3f})'.format(TAU)]
summary = []
for k, name in enumerate(names):
    summary.append({
        'metodo': name,
        'latencia_media_s': float(np.mean([rec['time'][k] for rec in records])),
        'pico_vram_mb': float(max(rec['vram'][k] for rec in records)),
        'passos_completos_medio': float(np.mean([rec['full'][k] for rec in records])),
        'clip_medio': float(np.mean([rec['clip'][k] for rec in records])),
        'lpips_medio': float(np.mean([rec['lpips'][k] for rec in records])),
    })
base_mean = summary[0]['latencia_media_s']
for row in summary:
    row['speedup'] = base_mean / row['latencia_media_s']

lines = [
    '| Método | Latência média (s) | Speedup | Pico de VRAM (MB) | Passos completos executados (de 50) | CLIP Score médio | LPIPS médio vs. sem cache |',
    '|---|---:|---:|---:|---:|---:|---:|',
]
for row in summary:
    lines.append('| {} | {:.3f} | {:.2f}x | {:.1f} | {:.1f} | {:.4f} | {:.4f} |'.format(
        row['metodo'], row['latencia_media_s'], row['speedup'], row['pico_vram_mb'],
        row['passos_completos_medio'], row['clip_medio'], row['lpips_medio']))

print('Tabela final para o relatório (colar na Seção 4 de docs/respostas_template.md):')
print()
for line in lines:
    print(line)
print()
print(f'Scheduler: {SCHEDULER_NAME} | tau escolhido: {TAU}')

(RESULTS_DIR / 'tabela_resultados.md').write_text('\n'.join(lines) + '\n', encoding='utf-8')
with (RESULTS_DIR / 'metricas_resultados.csv').open('w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow([
        'prompt',
        'sem_cache_s', 'oficial_s', 'adaptativo_s',
        'sem_cache_vram_mb', 'oficial_vram_mb', 'adaptativo_vram_mb',
        'sem_cache_passos', 'oficial_passos', 'adaptativo_passos',
        'sem_cache_clip', 'oficial_clip', 'adaptativo_clip',
        'oficial_lpips', 'adaptativo_lpips',
    ])
    for rec in records:
        writer.writerow([rec['prompt'], *rec['time'], *rec['vram'], *rec['full'], *rec['clip'], *rec['lpips'][1:]])

(RESULTS_DIR / 'resumo_resultados.json').write_text(
    json.dumps({
        'scheduler': SCHEDULER_NAME,
        'tau_escolhido': TAU,
        'calibracao': calibration_rows,
        'avaliacao': summary,
    }, ensure_ascii=False, indent=2),
    encoding='utf-8',
)

thumb = 256
row_h = thumb + 90
canvas = Image.new('RGB', (thumb * 3, row_h * len(records)), 'white')
draw = ImageDraw.Draw(canvas)
for i, rec in enumerate(records):
    y = i * row_h
    draw.text((10, y + 8), 'Prompt {}: {}'.format(i + 1, rec['prompt'][:95]), fill='black')
    draw.text((10, y + 48), 'Sem cache: {:.2f}s'.format(rec['time'][0]), fill='black')
    draw.text((thumb + 10, y + 48), 'Oficial: {:.2f}s'.format(rec['time'][1]), fill='black')
    draw.text((2 * thumb + 10, y + 48), 'Adaptativo: {:.2f}s'.format(rec['time'][2]), fill='black')
    for col, img in enumerate(rec['images']):
        canvas.paste(img.resize((thumb, thumb)), (col * thumb, y + 90))
canvas.save(RESULTS_DIR / 'comparacao_visual.png')
display(canvas)
print('Arquivos salvos em', RESULTS_DIR)